In [ ]:
import torch
import torch.nn as nn
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler


In [ ]:
#load trained model
def load_model():
    model = AE(input_size=105, latent_size=10)
    model.load_state_dict(
        torch.load("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/normalized_autoencoder")
    )
    model.eval()
    return model
load_model()

In [ ]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data"
unseen_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/unseen_data/unseen_data.csv"

euler_nr = pd.read_csv(os.path.join(path, "euler_nr_data.csv"))
datatable = pd.read_csv(os.path.join(path, "Final_data.csv"))

euler_nr = euler_nr[['SubjectID', 'avg_euler_centered_neg_sqrt']].copy()
df = datatable.merge(euler_nr, how="right")
df = df.iloc[:, 2:]

#create dataframe without the last few columns
df_for_blr = df.iloc[:,0:112]
df_unseen = df_for_blr[:3000]
# df_unseen.to_csv(unseen_path)
df_for_blr = df_for_blr[3000:]

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df_for_blr = df_for_blr.drop(columns=['SubjectID','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities'])


In [ ]:
train_df, test_df = train_test_split(df_for_blr, test_size=0.2, random_state=1107)

scaler = RobustScaler()
train_scaled = scaler.fit_transform(train_df)
test_scaled = scaler.transform(test_df)

train_tensor = torch.tensor(train_scaled, dtype=torch.float32)
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

training_data = TensorDataset(train_tensor)
test_data = TensorDataset(test_tensor)

train_loader = DataLoader(training_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [ ]:
# class VAE(nn.Module):
#     def __init__(self, input_size=111,latent_size=10):
#         super(VAE, self).__init__()
#         self.input_size = input_size
#         self.latent_size = latent_size

#         #  encoder network
#         self.encoder_part1 = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(input_size, 56), nn.ReLU(),
#             nn.Linear(56, 28), nn.ReLU(),
#             nn.Linear(28, latent_size), nn.ReLU()
#         )
#         self.encoder_mean   = nn.Linear(16, latent_size)
#         self.encoder_logvar = nn.Linear(16, latent_size)

#         # decoder network
#         self.decoder_part1 = nn.Linear(latent_size, input_size)
#         self.decoder_part2 = nn.Sequential(
#             nn.ReLU(),
#             nn.Linear(latent_size, 28), nn.ReLU(),
#             nn.Linear(28, 56), nn.ReLU(),
#             nn.Linear(56, input_size),
#             nn.Sigmoid()
#         )

#     def encode(self, x):
#         h = self.encoder_part1(x)
#         return self.encoder_mean(h), self.encoder_logvar(h)

#     def sample_latent(self, mean_z, logvar_z):
#         eps = torch.randn_like(mean_z)
#         std_z = torch.exp(0.5 * logvar_z)
#         return mean_z + eps * std_z

#     def decode(self, z):
#         h = self.decoder_part1_z(z)
#         return self.decoder_part2(h)

#     def forward(self, x):
#         mean_z, logvar_z = self.encode(x)
#         z = self.sample_latent(mean_z, logvar_z)
#         return self.decode(z), mean_z, logvar_z


In [ ]:
class AE(nn.Module):
    def __init__(self, input_size=108, latent_size=10):
        super(AE, self).__init__()
        self.input_size = input_size
        self.latent_size = latent_size

        # encoder network
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 56), nn.ReLU(),
            nn.Linear(56, 28), nn.ReLU(),
            nn.Linear(28, latent_size)
        )
        # decoder network
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, 28), nn.ReLU(),
            nn.Linear(28, 56), nn.ReLU(),
            nn.Linear(56, input_size)
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)
        
model = load_model()

In [ ]:
def train_ae(model, train_loader, test_loader, epochs, lr):
    loss_func = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    

    for epoch in range(epochs):
        
        ################
        #### TRAIN #####
        ################
        
        model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            optimizer.zero_grad()
            x = batch[0]
            prediction = model(x)
            loss = loss_func(prediction, x)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss_average = train_loss / len(train_loader)
        train_losses.append(train_loss_average)

        
        ################
        ####  TEST #####
        ################
        
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for x, in test_loader:
                prediction = model(x)
                loss = loss_func(prediction, x)
                test_loss += loss.item()
        test_loss_average = test_loss / len(test_loader)
        test_losses.append(test_loss_average)
        
        print(f'Epoch: {epoch+1}')
        print(f'Train loss: {train_loss_average}')
        print(f'Test loss: {test_loss_average}')
        print('\n')

    return train_losses, test_losses
                

In [ ]:
def plot_losses(train_losses, test_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title('Training and Test Loss')
    plt.legend()
    plt.show()


lr = 0.001
epochs = 100
input_size = df_for_blr.shape[1]
model = AE(input_size=input_size, latent_size=10)

train_losses, test_losses = train_ae(model, train_loader, test_loader, epochs=epochs, lr=lr)

In [ ]:
plot_losses(train_losses, test_losses)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate_reconstruction(model, dataloader, device):
    model.eval()
    all_outputs = []
    all_inputs = []

    with torch.no_grad():
        for batch in dataloader:
            x, = batch
            x = x.to(device)
            output = model(x)
            all_outputs.append(output.cpu().numpy())
            all_inputs.append(x.cpu().numpy())

    all_outputs = np.concatenate(all_outputs, axis=0)
    all_inputs = np.concatenate(all_inputs, axis=0)

    mse = mean_squared_error(all_inputs, all_outputs)
    mae = mean_absolute_error(all_inputs, all_outputs)
    r2 = r2_score(all_inputs, all_outputs)

    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R² Score: {r2:.4f}")

    return mse, mae, r2

model = load_model()
mse, mae, r2 = evaluate_reconstruction(model, test_loader, device)


In [ ]:
def collect_all_data(model, dataloader):
    model.eval()
    all_inputs = []
    all_outputs = []

    with torch.no_grad():
        for batch, in dataloader:
            x = batch
            output = model(x)

            all_inputs.append(x.cpu().numpy())
            all_outputs.append(output.cpu().numpy())

    all_inputs = np.concatenate(all_inputs, axis=0)
    all_outputs = np.concatenate(all_outputs, axis=0)

    return all_inputs, all_outputs


def plot_feature_errors(feature_error_df):
    
    #MAE
    plt.figure(figsize=(20, 6))
    plt.bar(feature_error_df["feature"], feature_error_df["MAE"],color="orangered")
    plt.xticks(rotation=90, ha="right")
    plt.xlabel("Feature")
    plt.ylabel("Mean Absolute Error")
    plt.title(f"Features by Reconstruction Error (MAE)")
    plt.tight_layout()
    plt.show()

    #MSE
    plt.figure(figsize=(20, 6))
    plt.bar(feature_error_df["feature"], feature_error_df["MSE"], color="purple")
    plt.xticks(rotation=90, ha="right")
    plt.xlabel("Feature")
    plt.ylabel("Mean Squared Error")
    plt.title(f"Features by Reconstruction Error (MSE)")
    plt.tight_layout()
    plt.show()

    feature_error_df = feature_error_df.sort_values("R2", ascending=False)
    
    #R2
    plt.figure(figsize=(25, 10))
    plt.bar(feature_error_df["feature"], feature_error_df["R2"], color="deeppink")
    plt.xticks(rotation=90, ha="right")
    plt.xlabel("Feature")
    plt.ylabel("R2")
    plt.title("R2 per feature")
    plt.tight_layout()
    plt.show()

def plot_features(inputs_original, outputs_original, feature_names, feature_name, bins=50):

    idx = feature_names.index(feature_name)

    actual = inputs_original[:, idx]
    predicted = outputs_original[:, idx]
    error = predicted - actual
    
    mae = np.mean(np.abs(error))
    mse = np.mean(error ** 2)
    r2 = r2_score(actual, predicted)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # actual vs prediction
    axes[0].hist(actual, bins=bins, alpha=0.6, label="Actual", color="gold")
    axes[0].hist(predicted, bins=bins, alpha=0.6, label="Prediction", color="purple")
    axes[0].set_xlabel(feature_name)
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"{feature_name}: Actual vs Reconstruction\nR² = {r2:.3f}")
    axes[0].legend()

    # mae & mse
    axes[1].hist(error, bins=bins, color="coral")
    axes[1].set_xlabel("Reconstruction Error")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Error Distribution\nMAE = {mae:.4f} | MSE = {mse:.4f}")

    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import r2_score
feature_names = df_for_blr.columns.to_list()
num_features = len(feature_names)

inputs_scaled, outputs_scaled = collect_all_data(model, test_loader)

inputs_orig = scaler.inverse_transform(inputs_scaled)
outputs_orig = scaler.inverse_transform(outputs_scaled)

errors = outputs_orig - inputs_orig
abs_errors = np.abs(errors)

feature_error_df = pd.DataFrame({
    "feature": feature_names,
    "MAE": abs_errors.mean(axis=0),
    "MSE": (errors ** 2).mean(axis=0)
}).sort_values("MAE", ascending=False)


r2_scores = [
    r2_score(inputs_orig[:, i], outputs_orig[:, i])
    for i in range(len(feature_names))
]

feature_error_df["R2"] = r2_scores

plot_feature_errors(feature_error_df)


In [ ]:
for feature in feature_names:
    plot_features(inputs_orig, outputs_orig, feature_names, feature, bins=50)

In [ ]:
torch.save(model.state_dict(), "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/normalized_autoencoder")


In [ ]:
load_model()